In [2]:
import os
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.ensemble import IsolationForest
from dataclasses import dataclass
from enum import Enum
from typing import Optional, List, Tuple

# =============================================================================
# 1. ENUMS AND CONFIGURATION 
# =============================================================================
class FlagClass(Enum):
    VALID = "VALID"
    IMPLAUSIBLE = "IMPLAUSIBLE"

@dataclass
class FlagResult:
    raw_value: float
    flag: FlagClass
    reasons: List[str]

@dataclass
class ChannelConfig:
    tau: float                  
    tau_uncertainty_margin: float 
    T_sink: float               
    T_drive_max: float          
    sensor_min: float           
    sensor_max: float           

# =============================================================================
# 2. PHYSICS HELPER EQUATIONS 
# =============================================================================
def calc_alpha(dt: float, tau: float) -> float:
    """Discrete exponential filter coefficient."""
    return math.exp(-dt / tau)

def calc_slope_bounds(T: float, T_sink: float, T_drive_max: float, tau: float) -> Tuple[float, float]:
    """Rate of change boundaries (dT/dt) based on sensor thermal mass (K/s)."""
    max_slope = (T_drive_max - T) / tau
    min_slope = -(T - T_sink) / tau
    return min_slope, max_slope

def calc_radiation_error(T_tc: float, T_w: float, h: float, epsilon: float = 0.6) -> float:
    """Radiation bias (K). Diagnostic only."""
    sigma = 5.67e-8
    return (epsilon * sigma * (T_tc**4 - T_w**4)) / h

def calc_conduction_error(T_mount: float, T_g: float, h: float, P: float, k: float, Ac: float, L: float) -> float:
    """Conduction (stem) error (K)."""
    m = math.sqrt((h * P) / (k * Ac))
    return (T_mount - T_g) / math.cosh(m * L)

def calc_recovery_effect(T_static: float, u: float, cp: float, r: float = 0.75) -> float:
    """Velocity recovery effect (K). Relevant > 50 m/s."""
    return T_static + (r * u**2) / (2 * cp)

def calc_biot_number(h: float, L_c: float, k_s: float) -> float:
    """Biot number to validate lumped capacitance model. Warn if > 0.1."""
    return (h * L_c) / k_s

def calc_quantisation_noise(resolution: float) -> float:
    """Lower bound for noise scale based on quantization."""
    return resolution / math.sqrt(12)

def calc_daq_bandwidth(nplc: int, f_mains: float = 50.0) -> float:
    """Effective bandwidth (Hz) of DAQ970A moving-average integration."""
    t_int = nplc / f_mains
    return 0.44 / t_int

# =============================================================================
# 3. STREAMING PHYSICS ENGINE
# =============================================================================
class TemperatureChannel:
    def __init__(self, config: ChannelConfig):
        self.config = config
        self.t_prev: Optional[float] = None
        self.T_prev: Optional[float] = None

    def process(self, t_now: float, T_raw: float) -> FlagResult:
        flags = []
        is_vetoed = False

        # 1. Absolute Limits
        if T_raw < self.config.sensor_min or T_raw > self.config.sensor_max:
            is_vetoed = True
            flags.append("Hard sensor limit violated")

        if T_raw < self.config.T_sink or T_raw > self.config.T_drive_max:
            is_vetoed = True
            flags.append("Thermodynamic bound violated")

        # 2. Dynamic Limits
        dt = (t_now - self.t_prev) if self.t_prev is not None else None
        
        if self.T_prev is not None and dt is not None and dt > 0:
            dT_dt = (T_raw - self.T_prev) / dt
            min_slope, max_slope = calc_slope_bounds(
                self.T_prev, self.config.T_sink, self.config.T_drive_max, self.config.tau
            )
            max_allowed = max_slope * self.config.tau_uncertainty_margin
            min_allowed = min_slope * self.config.tau_uncertainty_margin

            if dT_dt > max_allowed or dT_dt < min_allowed:
                is_vetoed = True
                flags.append(f"Physics Rate Violation ({dT_dt:.2f} K/s)")

        final_class = FlagClass.IMPLAUSIBLE if is_vetoed else FlagClass.VALID

        # Architectural Rule: Vetoed data does not update baseline state
        if not is_vetoed:
            self.t_prev = t_now
            self.T_prev = T_raw

        return FlagResult(raw_value=T_raw, flag=final_class, reasons=flags)

# =============================================================================
# 4. HYBRID BATCH PROCESSING & VISUALIZATION
# =============================================================================
def process_and_plot_file(file_path: str, config: ChannelConfig):
    if not os.path.exists(file_path):
        print(f"Error: File not found -> {os.path.basename(file_path)}")
        return

    # 1. Parse Data (Bypassing Metadata)
    header_idx = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if 'Scan Swee' in line or 'Scan Number' in line or '101Time' in line:
                header_idx = idx
                break
                
    df = pd.read_csv(file_path, skiprows=header_idx)
    df = df.dropna(how='all', axis=1)
    
    time_cols = [col for col in df.columns if 'time' in col.lower()]
    time_col = time_cols[0] if time_cols else df.columns[0]
    sensor_cols = [col for col in df.columns if '°c' in col.lower() or '101' in col]
    sensor_col = sensor_cols[-1] if sensor_cols else df.columns[-1]
    
    df[sensor_col] = pd.to_numeric(df[sensor_col], errors='coerce')
    
    # Adaptive Time Parsing
    first_time_val = str(df[time_col].dropna().iloc[0])
    if ' ' in first_time_val or '-' in first_time_val:
        df['Timestamp'] = pd.to_datetime(df[time_col], errors='coerce')
        df_clean = df.dropna(subset=['Timestamp', sensor_col]).copy()
        df_clean['Seconds_Elapsed'] = (df_clean['Timestamp'] - df_clean['Timestamp'].iloc[0]).dt.total_seconds()
    else:
        df['Timestamp'] = pd.to_timedelta("00:" + df[time_col].astype(str), errors='coerce')
        df_clean = df.dropna(subset=['Timestamp', sensor_col]).copy()
        df_clean['Seconds_Elapsed'] = df_clean['Timestamp'].dt.total_seconds()

    # 2. RUN PHYSICS ENGINE
    engine = TemperatureChannel(config)
    physics_flags = []
    physics_reasons = []
    
    for _, row in df_clean.iterrows():
        res = engine.process(row['Seconds_Elapsed'], row[sensor_col])
        physics_flags.append(res.flag.value)
        physics_reasons.append(" | ".join(res.reasons))
        
    df_clean['Physics_Flag'] = physics_flags
    df_clean['Physics_Reason'] = physics_reasons

    # 3. RUN ISOLATION FOREST (Machine Learning)
    # Calculate a raw dT/dt array as a secondary feature for the ML model
    df_clean['dT_dt_raw'] = df_clean[sensor_col].diff() / df_clean['Seconds_Elapsed'].diff()
    df_clean['dT_dt_raw'] = df_clean['dT_dt_raw'].fillna(0)
    
    # Isolate features and fit the model (contamination = expected anomaly rate, e.g., 1%)
    ml_features = df_clean[[sensor_col, 'dT_dt_raw']]
    iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
    
    # Predict returns 1 for inliers, -1 for outliers
    df_clean['ML_Outlier'] = iso_forest.fit_predict(ml_features)

    # 4. SUMMARY METRICS
    phys_anomalies = df_clean[df_clean['Physics_Flag'] == 'IMPLAUSIBLE']
    ml_anomalies = df_clean[(df_clean['ML_Outlier'] == -1) & (df_clean['Physics_Flag'] == 'VALID')]
    
    total_pts = len(df_clean)
    print(f"\n{'='*60}")
    print(f"FILE: {os.path.basename(file_path)}")
    print(f"{'-'*60}")
    print(f"Total Data Points       : {total_pts}")
    print(f"Physics Engine Vetoes   : {len(phys_anomalies)} (Impossible events)")
    print(f"Isolation Forest Outliers: {len(ml_anomalies)} (Statistical anomalies)")
    print(f"{'='*60}\n")
    
    # 5. GENERATE PLOT
    fig = go.Figure()
    
    # Base Signal Line
    fig.add_trace(go.Scatter(
        x=df_clean['Timestamp'], y=df_clean[sensor_col], 
        mode='lines', name='Raw Trajectory',
        line=dict(color='lightgrey', width=1.5), hoverinfo='skip'
    ))
    
    # Valid Data
    valid_data = df_clean[(df_clean['Physics_Flag'] == 'VALID') & (df_clean['ML_Outlier'] == 1)]
    fig.add_trace(go.Scatter(
        x=valid_data['Timestamp'], y=valid_data[sensor_col], 
        mode='markers', name='Normal Operation',
        marker=dict(color='#1f77b4', size=4)
    ))

    # Isolation Forest Outliers (Orange Circles)
    if not ml_anomalies.empty:
        fig.add_trace(go.Scatter(
            x=ml_anomalies['Timestamp'], y=ml_anomalies[sensor_col], 
            mode='markers', name='ML Statistical Outlier',
            marker=dict(color='orange', size=6, symbol='circle', line=dict(width=1, color='darkorange')),
            hovertemplate="Time: %{x}<br>Temp: %{y}°C<br>Flag: Isolation Forest Outlier<extra></extra>"
        ))
    
    # Physics Engine Vetoes (Red X) - Plotted last so they sit on top
    if not phys_anomalies.empty:
        fig.add_trace(go.Scatter(
            x=phys_anomalies['Timestamp'], y=phys_anomalies[sensor_col], 
            mode='markers', name='Physics Veto (Impossible)',
            marker=dict(color='red', size=8, symbol='x', line=dict(width=1.5, color='darkred')),
            text=phys_anomalies['Physics_Reason'],
            hovertemplate="Time: %{x}<br>Temp: %{y}°C<br>Reason: %{text}<extra></extra>"
        ))
        
    fig.update_layout(
        title=f"Hybrid Anomaly Detection (Physics + Isolation Forest): {os.path.basename(file_path)}",
        xaxis_title="Time",
        yaxis_title="Temperature (°C)",
        template='plotly_white',
        hovermode='x unified',
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
        height=500, width=1200
    )
    
    fig.show()

# =============================================================================
# 5. EXECUTION LOOP
# =============================================================================
file_paths = [
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\2 2025-11-21 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\03-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\03-11-2025 2 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\06-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\07-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\13-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\14-11-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\15-09-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\16-09-2025 0 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\17-11-2025.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\18-09-2025 0.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\21-11-2025 0 2025-11-21.csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\2025-11-15 .csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\2025-11-17 .csv",
    r"C:\Users\hari7\Documents\Anamoly Detection\OneDrive_1_10-6-2026\2025-11-19  0.csv"
]

shared_config = ChannelConfig(
    tau=15.0, 
    tau_uncertainty_margin=1.2, 
    T_sink=20.0, 
    T_drive_max=2200.0, 
    sensor_min=-50.0, 
    sensor_max=1370.0
)

for fp in file_paths:
    process_and_plot_file(fp, shared_config)


FILE: 2 2025-11-21 0.csv
------------------------------------------------------------
Total Data Points       : 2770
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 28 (Statistical anomalies)




FILE: 03-11-2025 0.csv
------------------------------------------------------------
Total Data Points       : 893
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 9 (Statistical anomalies)




FILE: 03-11-2025 2 0.csv
------------------------------------------------------------
Total Data Points       : 735
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 8 (Statistical anomalies)




FILE: 06-11-2025 0.csv
------------------------------------------------------------
Total Data Points       : 2664
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 27 (Statistical anomalies)




FILE: 07-11-2025 0.csv
------------------------------------------------------------
Total Data Points       : 2805
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 29 (Statistical anomalies)



...


FILE: 18-09-2025 0.csv
------------------------------------------------------------
Total Data Points       : 910
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 10 (Statistical anomalies)




FILE: 21-11-2025 0 2025-11-21.csv
------------------------------------------------------------
Total Data Points       : 10497
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 105 (Statistical anomalies)




FILE: 2025-11-15 .csv
------------------------------------------------------------
Total Data Points       : 884
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 9 (Statistical anomalies)




FILE: 2025-11-17 .csv
------------------------------------------------------------
Total Data Points       : 424
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 5 (Statistical anomalies)




FILE: 2025-11-19  0.csv
------------------------------------------------------------
Total Data Points       : 1255
Physics Engine Vetoes   : 0 (Impossible events)
Isolation Forest Outliers: 13 (Statistical anomalies)

